# 🦜 Bird Species Identifier

Upload any bird image and all trained classical ML models (SVM, k-NN, Decision Tree, Random Forest, Logistic Regression) will identify the species using frozen ResNet-18 deep visual features.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# ---- Setup: Install and import dependencies ----
import os, sys, shutil
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

import torch
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms

MODELS_DIR = Path('./models')
DATA_DIR   = Path('./processed_data')

# ---- Load label mapping ----
label_map_path = DATA_DIR / 'label_mapping.pkl'
if not label_map_path.exists():
    raise FileNotFoundError('label_mapping.pkl not found. Please run 07_Model_Comparison.ipynb first to extract features and train all models.')

label_mapping = joblib.load(label_map_path)
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]
print(f'Loaded {len(class_names)} species classes.')
print('Classes:', class_names)

In [ ]:
# ---- Load all trained ML models ----
model_files = {
    'SVM':                'svm_model.pkl',
    'k-NN':               'knn_model.pkl',
    'Decision Tree':      'decision_tree_model.pkl',
    'Random Forest':      'random_forest_model.pkl',
    'Logistic Regression':'logistic_regression_model.pkl'
}

models = {}
for name, fname in model_files.items():
    p = MODELS_DIR / fname
    if p.exists():
        models[name] = joblib.load(p)
        print(f'  Loaded: {name}')
    else:
        print(f'  [Warning] Model not found: {fname} — run 03-06 notebooks to train it.')

print(f'\n{len(models)} model(s) ready for identification.')

In [ ]:
# ---- Load frozen ResNet-18 feature extractor ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

weights = tv_models.ResNet18_Weights.DEFAULT
resnet = tv_models.resnet18(weights=weights)
resnet.fc = torch.nn.Identity()  # 512-dim output
resnet.eval()
resnet.to(device)

transform = tv_transforms.Compose([
    tv_transforms.Resize((224, 224)),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
])

# Load the scaler fitted during training
from sklearn.preprocessing import StandardScaler
import numpy as np

deep_npz = DATA_DIR / 'deep_features.npz'
if deep_npz.exists():
    _d = np.load(deep_npz)
    scaler = StandardScaler()
    scaler.fit(_d['X_train'])
    print('Scaler fitted on ResNet-18 training features.')
else:
    scaler = None
    print('[Warning] deep_features.npz not found. Features will NOT be standardized.')

def extract_deep_features(img: Image.Image) -> np.ndarray:
    tensor = transform(img.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(tensor).squeeze(0).cpu().numpy()
    feat = feat.reshape(1, -1).astype(np.float32)
    if scaler is not None:
        feat = scaler.transform(feat)
    return feat

print('ResNet-18 feature extractor ready.')

In [ ]:
# ---- Upload your bird image ----
from google.colab import files as colab_files

print('Please upload a bird image (JPG/PNG)...')
uploaded = colab_files.upload()

if not uploaded:
    raise ValueError('No file uploaded. Please upload a bird image.')

img_filename = list(uploaded.keys())[0]
img = Image.open(img_filename).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f'Uploaded Image: {img_filename}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Image size: {img.size}')

In [ ]:
# ---- Predict bird species using all trained ML models ----
if not models:
    raise RuntimeError('No models loaded. Run notebooks 03-06 first to train and save the classifiers.')

features = extract_deep_features(img)

predictions = {}
probabilities = {}

for model_name, clf in models.items():
    # Handle Pipeline (SVM might be wrapped)
    pred_idx = clf.predict(features)[0]
    pred_species = class_names[pred_idx] if pred_idx < len(class_names) else f'Unknown (class {pred_idx})'
    predictions[model_name] = pred_species

    # Get probability / confidence if available
    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(features)[0]
        probabilities[model_name] = round(float(np.max(proba)) * 100, 1)
    else:
        probabilities[model_name] = None

# ---- Print results ----
print('\n' + '='*55)
print('         BIRD SPECIES IDENTIFICATION RESULTS')
print('='*55)
for model_name, species in predictions.items():
    conf = f'  (Confidence: {probabilities[model_name]:.1f}%)' if probabilities[model_name] else ''
    print(f'  {model_name:<22}: {species}{conf}')
print('='*55)

# Majority vote
from collections import Counter
vote_counts = Counter(predictions.values())
majority_species = vote_counts.most_common(1)[0][0]
print(f'\n🏆 MAJORITY VOTE PREDICTION: {majority_species}')

In [ ]:
# ---- Visualize predictions as a summary chart ----
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

model_names = list(predictions.keys())
species_preds = list(predictions.values())
unique_species = sorted(set(species_preds))

# Color map for species
cmap = plt.cm.get_cmap('tab10', len(unique_species))
species_colors = {sp: cmap(i) for i, sp in enumerate(unique_species)}
bar_colors = [species_colors[s] for s in species_preds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: uploaded image
axes[0].imshow(img)
axes[0].set_title('Uploaded Bird Image', fontsize=13, fontweight='bold')
axes[0].axis('off')

# Right: model predictions bar
y_pos = np.arange(len(model_names))
bars = axes[1].barh(y_pos, [1]*len(model_names), color=bar_colors, edgecolor='gray')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(model_names, fontsize=11)
axes[1].set_xticks([])
axes[1].set_title('Model Predictions', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

for i, (bar, sp) in enumerate(zip(bars, species_preds)):
    conf_text = f' ({probabilities[model_names[i]]:.1f}%)' if probabilities[model_names[i]] else ''
    axes[1].text(0.02, bar.get_y() + bar.get_height()/2,
                 f'{sp}{conf_text}', va='center', fontsize=10, color='black', fontweight='bold')

# Legend for species colors
patches = [mpatches.Patch(color=species_colors[sp], label=sp) for sp in unique_species]
axes[1].legend(handles=patches, loc='lower right', fontsize=9, title='Predicted Species')

plt.suptitle(f'🏆 Majority Vote: {majority_species}', fontsize=15, fontweight='bold', color='darkgreen')
plt.tight_layout()
plt.show()